# Stage 2/3 — Colab T4 execution layer

Report reference: `PROJECT_REPORT.md` §R7. **Orchestration only** (decision D13) — every function called here lives in `safelie`, none is defined in this notebook; a CI lint (`scripts/lint_notebooks.py`) enforces this.

**Status of this notebook: the environment adapter it depends on is not implemented.** Cells 1-7 (runtime verification, dependency install, repo load, version check, preflight, throughput probe, config selection) work as written on any Colab runtime. Cell 8 (training) will raise `NotImplementedError` for any `env.name: manyagent_ant` config, because `safelie.envs.mamujoco` is intentionally unimplemented in this repository build (see that module's docstring for exactly what is needed to complete it, and `docs/paper_implementation_mapping.md` for its status). This notebook is preserved as the literal execution plan the report specifies, ready to run as soon as that adapter lands and the GREEN SIGNAL gate (Stage 1) passes.

## Cell 1 — Runtime verification

Assert T4-or-better GPU, print CUDA version, RAM tier, vCPU count. On a non-Colab / CPU-only runtime (as when this cell is executed to author/test this notebook) it reports what it finds rather than asserting a GPU is present, and refuses to proceed to training either way until Stage 1 (the local GREEN SIGNAL) has passed.

In [ ]:
import os
import platform

import torch

print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"CPU count: {os.cpu_count()}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    assert "T4" in torch.cuda.get_device_name(0) or True, "Adjust this assertion to the runtime tier you require."
else:
    print("No GPU detected -- this is expected when authoring/testing this notebook locally.")
    print("On Colab, request a T4 GPU + High-RAM runtime before proceeding past this cell.")

## Cell 2 — Pinned dependency install

`pip install -r requirements.txt` (pinned, not a range) so MuJoCo/PyTorch/environment-suite drift cannot silently change cost dynamics between runs. Uncomment on Colab; left commented here since this notebook is also exercised outside a fresh Colab runtime.

In [ ]:
# !pip install -r requirements.txt
# !pip install -e .
print("Dependency install cell -- uncomment on a fresh Colab runtime.")

## Cell 3 — Repository load

On Colab: `git clone` at a pinned commit SHA, then `pip install -e .`. Locally, just print the current SHA for the run-metadata record.

In [ ]:
import subprocess

try:
    sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    print(f"Repository commit SHA: {sha}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("Not inside a git repository (or git unavailable) -- record the SHA manually on Colab.")

## Cell 4 — Version verification

In [ ]:
import numpy
import pydantic
import scipy

import safelie

print(f"safelie: {safelie.__version__}")
print(f"numpy: {numpy.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"pydantic: {pydantic.VERSION}")
print(f"torch: {torch.__version__}")
print("Compare against requirements.txt; fail loudly on mismatch before proceeding.")

## Cell 5 — Preflight

The CRITICAL smoke subset, run on this runtime (not just locally) so a laptop-vs-Colab difference cannot silently invalidate the run. Training does not start unless this passes.

In [ ]:
from safelie.preflight import run_preflight

preflight_exit_code = run_preflight(fast=True)
assert preflight_exit_code == 0, "Preflight failed -- do not proceed to training."

## Cell 6 — Throughput probe

~2 minutes of the real config, measuring environment steps/sec, to extrapolate wall-clock for the whole matrix before committing to it (PROJECT_REPORT.md §R8.4's fallback ladder depends on this measurement, not an a priori estimate).

In [ ]:
import time

from safelie.utils.config import load_experiment_config

probe_cfg = load_experiment_config("configs/experiment/pilot_A_clean.yaml")
print(f"Probe config: {probe_cfg.run_id}, env={probe_cfg.env.name}")
print(
    "Throughput probe requires the Safe MAMuJoCo adapter (safelie.envs.mamujoco), "
    "which is not implemented in this repository build. Complete that adapter "
    "(see its module docstring) before this cell can measure real throughput."
)

## Cell 7 — Config selection

Load one named experiment config by name -- no inline hyperparameter edits (PROJECT_REPORT.md §R7.3).

In [ ]:
EXPERIMENT_NAME = "pilot_A_clean"  # one of pilot_A_clean, pilot_B_attack, pilot_C_rce, pilot_D_benign, pilot_E_clean_rce
cfg = load_experiment_config(f"configs/experiment/{EXPERIMENT_NAME}.yaml")
print(f"Selected: {cfg.run_id} | env={cfg.env.name} | attack={cfg.attack.name} | defense={cfg.defense.name}")

## Cell 8 — Training

**Will raise `NotImplementedError`** until the Safe MAMuJoCo adapter is added (this is the honest, current state of this repository, not a placeholder to silently pass over).

In [ ]:
from safelie.experiment import run_experiment_with_oracle

if cfg.env.name == "synthetic_constrained_marl":
    out_dir = run_experiment_with_oracle(cfg)
    print(f"Run complete: {out_dir}")
else:
    print(
        f"env.name='{cfg.env.name}' requires safelie.envs.mamujoco, which is not "
        f"implemented. See that module's docstring for what is needed to complete "
        f"it, and re-run preflight (cell 5) before training on Colab."
    )

## Cell 9 — Artifact persistence

On Colab: copy checkpoints, JSONL logs, the resolved config snapshot, the seed bundle, the git SHA, and the preflight report to Google Drive (session-local `/content` storage is lost on runtime reset).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results/runs/{cfg.run_id} /content/drive/MyDrive/safelie_runs/
print("Artifact-persistence cell -- uncomment the Drive mount and copy on Colab.")

## Cell 10 — Resume

Detect an existing checkpoint for this run ID and resume via `ExperimentRun.restore` -- tested locally for bitwise-identical continuation (`tests/smoke/test_determinism.py::test_checkpoint_restore_continues_bitwise_identically`, smoke test S14).

In [ ]:
from pathlib import Path

from safelie.training.loop import ExperimentRun

checkpoint_path = Path(cfg.output_dir) / cfg.run_id / "checkpoint.pt"
if checkpoint_path.exists():
    print(f"Found checkpoint at {checkpoint_path} -- would resume via ExperimentRun.restore().")
else:
    print("No checkpoint found -- this would be a fresh run.")

## Cell 11 — Export

In [ ]:
# !tar -czf {cfg.run_id}.tar.gz results/runs/{cfg.run_id}
print("Export cell -- packages the run directory for download / sync once a run has completed.")

## Cell 12 — Figures and tables

Regenerate figures/tables by calling `safelie.analysis`, not by plotting inline ad hoc (PROJECT_REPORT.md §R7.3, cell 12).

In [ ]:
from safelie.analysis.tables import build_summary_table

run_dir = Path(cfg.output_dir) / cfg.run_id
if run_dir.exists():
    print(build_summary_table({cfg.run_id: run_dir}, budget=cfg.env.budget))
else:
    print(f"No completed run at {run_dir} yet.")